# Atualização do Ambiente PIP

In [1]:
%pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# Passo 1 - Instalação das Bibliotecas

In [1]:
%pip install psycopg2-binary
%pip install pandas
%pip install minio
%pip install python-dotenv
%pip install sqlalchemy
%pip install parquet
%pip install pyarrow --quiet

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Passo 2 - Configurações e Importações

In [1]:
import os
import io
import psycopg2
import pandas as pd
from minio import Minio
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

# --- MinIO ---
MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'localhost:9000').replace('http://', '').replace('https://', '')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
BUCKET           = 'moedas'
CAMADA_ORIGEM    = 'bronze'
CAMADA_DESTINO   = 'silver'

# --- PostgreSQL ---
PG_CONFIG = {
    "dbname": os.getenv("PG_DB", "postgres"),
    "user": os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD", "postgres"),
    "host": os.getenv("PG_HOST", "localhost"),
    "port": os.getenv("PG_PORT", "5434")
}

TABELAS = ['extracao_moedas']

print('Configurações carregadas!')
print(f'   MinIO  : {MINIO_ENDPOINT}')
print(f'   Bucket : {BUCKET}')

Configurações carregadas!
   MinIO  : localhost:9000
   Bucket : moedas


# Passo 3 - Funções de Conecção do Minio

In [2]:
def get_minio_client():
    return Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=False
    )

def salvar_parquet_minio(df: pd.DataFrame, bucket: str, object_key: str):
    """Salva DataFrame como Parquet no MinIO sem arquivo local."""
    client = get_minio_client()

    if not client.bucket_exists(bucket):
        client.make_bucket(bucket)

    buf = io.BytesIO()
    df.to_parquet(buf, index=False, engine='pyarrow')
    buf.seek(0)
    tamanho = buf.getbuffer().nbytes

    client.put_object(
        bucket_name=bucket,
        object_name=object_key,
        data=buf,
        length=tamanho,
        content_type='application/octet-stream'
    )

    print(f'   -> MinIO: s3://{bucket}/{object_key}')
    print(f'   Linhas: {len(df):,} | Tamanho: {tamanho / 1024:.1f} KB')

# Testa conexão
try:
    client  = get_minio_client()
    buckets = [b.name for b in client.list_buckets()]
    print(f'MinIO OK — buckets disponíveis: {buckets}')
except Exception as e:
    print(f'MinIO Erro: {e}')

MinIO OK — buckets disponíveis: ['moedas']


# Passo 4 - Leitura dos Dados Brutos da Bronze

In [3]:
from sqlalchemy import create_engine
import pandas as pd

dfs_bronze = {}

def ler_bronze_postgres():
    # 1. Define a consulta SQL que faltava
    query = "SELECT * FROM public.extracao_moedas_bronze;"
    
    # 2. Cria a engine usando as configurações locais (porta 5434)
    engine = create_engine(
        f"postgresql://{PG_CONFIG['user']}:{PG_CONFIG['password']}@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['dbname']}"
    )

    # 3. Executa a leitura
    df = pd.read_sql(query, engine)
    return df

try:
    dfs_bronze['extracao_moedas'] = ler_bronze_postgres()
    print(f"Lendo [extracao_moedas]: {len(dfs_bronze['extracao_moedas']):,} registros carregados.")
except Exception as e:
    print(f"Erro ao ler tabela bronze: {e}")

Lendo [extracao_moedas]: 3 registros carregados.


# Passo 5 - Inspeção e Diagnóstico de Qualidade

In [4]:
def inspecionar_df(nome: str, df: pd.DataFrame):
    """Diagnóstico rápido de qualidade do DataFrame."""
    print(f'\n── {nome} ──')
    print(f'   Shape      : {df.shape}')
    print(f'   Tipos      : {df.dtypes.to_dict()}')
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if not nulos.empty:
        print(f'   Nulos      : {nulos.to_dict()}')
    else:
        print('   Nulos      : nenhum')
    print(f'   Duplicatas : {df.duplicated().sum()}')

for nome, df in dfs_bronze.items():
    inspecionar_df(nome, df)


── extracao_moedas ──
   Shape      : (3, 11)
   Tipos      : {'id_extracao': <StringDtype(na_value=nan)>, 'codigo_moeda': <StringDtype(na_value=nan)>, 'nome_moeda': <StringDtype(na_value=nan)>, 'valor_compra': dtype('float64'), 'valor_venda': dtype('float64'), 'alta': dtype('float64'), 'baixa': dtype('float64'), 'variacao': dtype('float64'), 'pct_mudanca': dtype('float64'), 'data_cotacao': dtype('<M8[us]'), 'data_atualizacao': dtype('<M8[us]')}
   Nulos      : nenhum
   Duplicatas : 0


# Passo 6 - Limpeza de dados Duplicados e Formatação (Silver Rules)

In [5]:
def limpar_datas(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Remove timezone e converte para datetime sem tz."""
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce').dt.tz_localize(None)
    return df

def remover_duplicatas(df: pd.DataFrame, subset: list = None) -> pd.DataFrame:
    """Remove linhas duplicadas e reporta quantas foram removidas."""
    antes  = len(df)
    df     = df.drop_duplicates(subset=subset)
    depois = len(df)
    if antes != depois:
        print(f'   Atenção: {antes - depois} duplicatas removidas')
    return df

COLUNAS_DATA = ['data_cotacao', 'data_atualizacao']
dfs_silver   = {}

# --- Processamento das Moedas ---
df = dfs_bronze['extracao_moedas'].copy()
df = remover_duplicatas(df, subset=['id_extracao', 'codigo_moeda'])
df = limpar_datas(df, COLUNAS_DATA)

# Conversão e Arredondamento dos Números
cols_numericas = ['valor_compra', 'valor_venda', 'alta', 'baixa', 'variacao', 'pct_mudanca']
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce').round(4)

df['nome_moeda'] = df['nome_moeda'].str.strip()
df['codigo_moeda'] = df['codigo_moeda'].str.strip().str.upper()

dfs_silver['extracao_moedas'] = df
print(f'extracao_moedas limpas : {len(df):,} registros')

extracao_moedas limpas : 3 registros


# Passo 7 - Enrriquecimento e Tabela Desnormalizada (moedas_completo)

In [6]:
df_moedas_completo = dfs_silver['extracao_moedas'].copy()

# Criando colunas calculadas analíticas (métricas de mercado)
df_moedas_completo['spread_venda_compra'] = (df_moedas_completo['valor_venda'] - df_moedas_completo['valor_compra']).round(4)
df_moedas_completo['amplitude_diaria'] = (df_moedas_completo['alta'] - df_moedas_completo['baixa']).round(4)

# Particionamento temporal
df_moedas_completo['ano'] = df_moedas_completo['data_atualizacao'].dt.year
df_moedas_completo['mes'] = df_moedas_completo['data_atualizacao'].dt.month
df_moedas_completo['dia'] = df_moedas_completo['data_atualizacao'].dt.day
df_moedas_completo['hora'] = df_moedas_completo['data_atualizacao'].dt.hour

colunas_finais = [
    'id_extracao', 'codigo_moeda', 'nome_moeda', 'valor_compra', 'valor_venda',
    'spread_venda_compra', 'alta', 'baixa', 'amplitude_diaria', 'variacao',
    'pct_mudanca', 'data_cotacao', 'data_atualizacao', 'ano', 'mes', 'dia', 'hora'
]

df_moedas_completo = df_moedas_completo[colunas_finais]

print(f'moedas_completo: {len(df_moedas_completo):,} registros | {len(df_moedas_completo.columns)} colunas')
df_moedas_completo.head(3)

moedas_completo: 3 registros | 17 colunas


,id_extracao,codigo_moeda,nome_moeda,valor_compra,valor_venda,spread_venda_compra,alta,baixa,amplitude_diaria,variacao,pct_mudanca,data_cotacao,data_atualizacao,ano,mes,dia,hora
0,20260829095359,USD,Dólar Americano/Real Brasileiro,5.1850,5.1860,0.001,5.2282,5.159,0.0692,0.0229,0.4436,2026-08-28 18:30:00,2026-08-29 09:53:59.946410,2026,8,29,9
1,20260829095359,EUR,Euro/Real Brasileiro,6.0056,6.0076,0.002,6.0561,5.987,0.0691,-0.0097,-0.1613,2026-08-28 19:16:08,2026-08-29 09:53:59.946410,2026,8,29,9
2,20260829095359,BTC,Bitcoin/Real Brasileiro,404830.0000,404831.0000,1.000,416221.0000,402496.000,13725.0000,-6447.0000,-1.5680,2026-08-29 09:50:00,2026-08-29 09:53:59.946410,2026,8,29,9


# Passo 8 - Validação de Nulos e Salvamento no Minio + PostgreSQL Silver

In [7]:
# 1. Verifica nulos
nulos = df_moedas_completo.isnull().sum()
nulos = nulos[nulos > 0]

if not nulos.empty:
    print('Nulos encontrados pós-tratamento:')
    print(nulos)
else:
    print('Nenhum valor nulo encontrado!')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 2. Salva no MinIO (Parquet)
print('\nSalvando moedas_completo no MinIO...')
salvar_parquet_minio(
    df_moedas_completo,
    BUCKET,
    f'{CAMADA_DESTINO}/moedas_completo/moedas_silver_{timestamp}.parquet'
)

# 3. Salva no PostgreSQL (Tabela Silver)
def salvar_postgres_silver(df: pd.DataFrame):
    from sqlalchemy import create_engine, text
    
    # 1. Conexão via SQLAlchemy Engine
    engine = create_engine(
        f"postgresql://{PG_CONFIG['user']}:{PG_CONFIG['password']}@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['dbname']}"
    )
    
    # 2. Garante que a estrutura da tabela existe
    query_ddl = """
    CREATE TABLE IF NOT EXISTS public.extracao_moedas_silver (
        id_extracao VARCHAR(50),
        codigo_moeda VARCHAR(10),
        nome_moeda VARCHAR(100),
        valor_compra NUMERIC(15, 4),
        valor_venda NUMERIC(15, 4),
        spread_venda_compra NUMERIC(15, 4),
        alta NUMERIC(15, 4),
        baixa NUMERIC(15, 4),
        amplitude_diaria NUMERIC(15, 4),
        variacao NUMERIC(15, 4),
        pct_mudanca NUMERIC(10, 4),
        data_cotacao TIMESTAMP,
        data_atualizacao TIMESTAMP,
        ano INT,
        mes INT,
        dia INT,
        hora INT,
        PRIMARY KEY (id_extracao, codigo_moeda)
    );
    """
    with engine.begin() as conn:
        conn.execute(text(query_ddl))
        # Esvazia a tabela sem apagar sua estrutura ou a View Gold dependente
        conn.execute(text("TRUNCATE TABLE public.extracao_moedas_silver CASCADE;"))
    
    # 3. Insere os dados atualizados
    df.to_sql(
        name='extracao_moedas_silver',
        con=engine,
        schema='public',
        if_exists='append',
        index=False
    )
    print('   -> Salvo no PostgreSQL (public.extracao_moedas_silver)!')

salvar_postgres_silver(df_moedas_completo)
print('\nSilver Moedas concluída com sucesso!')

Nenhum valor nulo encontrado!

Salvando moedas_completo no MinIO...
   -> MinIO: s3://moedas/silver/moedas_completo/moedas_silver_20260829_095615.parquet
   Linhas: 3 | Tamanho: 10.3 KB
   -> Salvo no PostgreSQL (public.extracao_moedas_silver)!

Silver Moedas concluída com sucesso!


# Passo 9 - Listagem e Auditoria dos Arquivos Gerados na Silver

In [8]:
client  = get_minio_client()
objetos = list(client.list_objects(BUCKET, prefix='silver/', recursive=True))

print(f'[{BUCKET}] Camada Silver — {len(objetos)} arquivo(s):\n')
for obj in sorted(objetos, key=lambda x: x.object_name):
    print(f'{obj.object_name:<75} {obj.size / 1024:>7.1f} KB')

[moedas] Camada Silver — 1 arquivo(s):

silver/moedas_completo/moedas_silver_20260829_095615.parquet                   10.3 KB
